# Insurance Charges — Exploratory Data Analysis & Preprocessing

**Dataset:** Medical Insurance (1338 records, 7 features)  
**Target:** `charges` (continuous — medical cost)  
**Flow:** Load → Inspect → Clean → Encode → Feature Engineer (BMI bins) → Chi-Square selection → Scale

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from scipy.stats import pearsonr, chi2_contingency

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 2. Load Data

In [ ]:
df = pd.read_csv('../datasets/insurance.csv')
print(df.shape)
df.head(10)

In [ ]:
df.info()
print('\n')
df.describe()

## 3. Univariate Analysis

In [ ]:
numeric_cols = ['age', 'bmi', 'children', 'charges']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col], kde=True, bins=50, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(f'{col} — outliers')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['children', 'sex', 'smoker']):
    sns.countplot(x=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Data Cleaning

In [ ]:
df_cleaned = df.copy()
print('Before dedup:', df_cleaned.shape)
df_cleaned.drop_duplicates(inplace=True)
print('After dedup: ', df_cleaned.shape)
print('Nulls:\n', df_cleaned.isnull().sum())

## 6. Encoding Categorical Features

In [ ]:
df_cleaned['sex']    = df_cleaned['sex'].map({'male': 1, 'female': 0})
df_cleaned['smoker'] = df_cleaned['smoker'].map({'yes': 1, 'no': 0})
df_cleaned.rename(columns={'sex': 'is_male', 'smoker': 'is_smoker'}, inplace=True)

df_cleaned = pd.get_dummies(df_cleaned, columns=['region'], drop_first=True)
df_cleaned = df_cleaned.astype(int)
df_cleaned.head()

## 7. Feature Engineering — BMI Categories

Standard WHO BMI ranges: Underweight <18.5, Normal 18.5–24.9, Overweight 25–29.9, Obese ≥30.

In [ ]:
df_cleaned['bmi_category'] = pd.cut(
    df_cleaned['bmi'],
    bins=[0, 18.5, 24.9, 29.9, float('inf')],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)
df_cleaned = pd.get_dummies(df_cleaned, columns=['bmi_category'], drop_first=True)
df_cleaned = df_cleaned.astype(int)
df_cleaned.head()

## 8. Feature Selection

In [ ]:
# Pearson correlation — numeric features vs charges
numeric_features = ['age', 'bmi', 'children', 'is_male', 'is_smoker']
correlations = {f: pearsonr(df_cleaned[f], df_cleaned['charges'])[0] for f in numeric_features}
corr_df = pd.DataFrame(correlations.items(), columns=['Feature', 'Pearson r'])
print(corr_df.sort_values('Pearson r', ascending=False).to_string(index=False))

In [ ]:
# Chi-Square test — binary features vs binned charges
cat_features = [
    'is_male', 'is_smoker',
    'region_northwest', 'region_southeast', 'region_southwest',
    'bmi_category_Normal', 'bmi_category_Overweight', 'bmi_category_Obese'
]

df_cleaned['charges_bin'] = pd.qcut(df_cleaned['charges'], q=4, labels=False)
alpha = 0.05
chi2_results = {}

for col in cat_features:
    contingency = pd.crosstab(df_cleaned[col], df_cleaned['charges_bin'])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency)
    decision = 'Keep' if p_val < alpha else 'Drop'
    chi2_results[col] = {'chi2': round(chi2_stat, 2), 'p_value': round(p_val, 4), 'Decision': decision}

chi2_df = pd.DataFrame(chi2_results).T.sort_values('p_value')
chi2_df

## 9. Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scale_cols = ['age', 'bmi']
df_cleaned[scale_cols] = scaler.fit_transform(df_cleaned[scale_cols])

# Move charges to last column
charges = df_cleaned.pop('charges')
df_cleaned['charges'] = charges
df_cleaned.drop(columns=['charges_bin'], inplace=True)

print('Final shape:', df_cleaned.shape)
df_cleaned.head()